In [0]:
%pip install websockets azure-eventhub

In [0]:
import asyncio
import json
import logging
from websockets import connect
from azure.eventhub.aio import EventHubProducerClient
from azure.eventhub import EventData

# Secrets (Databricks)
CONNECTION_STR = dbutils.secrets.get(
    scope="trading_secrets",
    key="eventhub_connection_string"
)

# Configuration
EVENTHUB_NAME = "klines-raw"
BATCH_SIZE    = 100
RETRY_DELAY_S = 5
SYMBOLS = [
    "btcusdt",   "ethusdt",   "bnbusdt",   "xrpusdt",   "solusdt",
    "adausdt",   "dogeusdt",  "avaxusdt",  "linkusdt",   "dotusdt",
    "trxusdt",   "ltcusdt",   "bchusdt",   "nearusdt",   "uniusdt",
    "atomusdt",  "etcusdt",   "filusdt",   "icpusdt",    "aptusdt",
    "arbusdt",   "opusdt",    "aaveusdt",  "stxusdt",    "vetusdt",
    "hbarusdt",  "algousdt",  "egldusdt",  "xtzusdt",    "axsusdt",
    "sandusdt",  "manausdt",  "thetausdt", "flowusdt",   "ksmusdt",
    "runeusdt",  "cakeusdt",  "compusdt",  "sushiusdt",  "snxusdt",
    "yfiusdt",   "zecusdt",   "dashusdt",  "neousdt",    "ontusdt",
    "zilusdt",   "enjusdt",   "batusdt",   "zrxusdt",    "storjusdt",
    "bandusdt",  "kncusdt",   "crvusdt",   "1inchusdt",  "celrusdt",
    "sklusdt",   "ankrusdt",  "ctsiusdt",  "rlcusdt",    "qtumusdt",
    "icxusdt",   "cotiusdt",  "glmusdt",   "ckbusdt",    "tlmusdt",
    "maskusdt",  "galausdt",  "imxusdt",   "woousdt",    "gmtusdt",
    "apeusdt",   "ldousdt",   "fetusdt",   "injusdt",    "suiusdt",
    "seiusdt",   "tiausdt",   "wldusdt",   "pythusdt",   "jupusdt",
    "wifusdt",   "bomeusdt",  "enausdt",   "taousdt",    "notusdt",
    "eigenusdt", "scrusdt",   "moveusdt",  "meusdt",     "virtualusdt",
]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("Producer")

# WebSocket URL builder
def build_ws_url(symbols: list[str]):
    streams = "/".join(f"{s}@kline_1m" for s in symbols)
    return f"wss://stream.binance.com:9443/stream?streams={streams}"


BINANCE_WS_URL = build_ws_url(SYMBOLS)
logger.info(f"Subscribing to {len(SYMBOLS)} symbols.")

# Producer
async def produce():
    client = EventHubProducerClient.from_connection_string(
        conn_str=CONNECTION_STR,
        eventhub_name=EVENTHUB_NAME,
    )

    async with client:
        while True:
            batch = None
            try:
                logger.info("Connecting to Binance WebSocket...")
                async with connect(
                    BINANCE_WS_URL,
                    ping_interval=30,
                    ping_timeout=10,
                    max_size=10 * 1024 * 1024,  
                ) as websocket:
                    logger.info(f"Connection established — {len(SYMBOLS)} streams active.")
                    batch = await client.create_batch()

                    async for message in websocket:
                        msg  = json.loads(message)
                        data = msg.get("data")

                        if not data or "k" not in data:
                            logger.warning(f"Unexpected format: {message[:120]}")
                            continue

                        k         = data["k"]
                        is_closed = k.get("x", False)

                        event_data = EventData(json.dumps(data))

                        # Batch size limit hit — flush and reset before adding
                        try:
                            batch.add(event_data)
                        except ValueError:
                            await client.send_batch(batch)
                            logger.info("Batch size limit — flushed and reset.")
                            batch = await client.create_batch()
                            batch.add(event_data)

                        # Send when batch reaches target size
                        if len(batch) >= BATCH_SIZE:
                            await client.send_batch(batch)
                            logger.info(
                                f"Batch sent | size={BATCH_SIZE} | "
                                f"symbol={k.get('s')} | closed={is_closed}"
                            )
                            batch = await client.create_batch()

            except Exception as e:
                logger.error(f"Connection error: {e}. Reconnecting in {RETRY_DELAY_S}s...")

                if batch and len(batch) > 0:
                    try:
                        await client.send_batch(batch)
                        logger.info(f"Flushed {len(batch)} messages before reconnect.")
                    except Exception as flush_err:
                        logger.error(f"Pre-reconnect flush failed: {flush_err}")

                await asyncio.sleep(RETRY_DELAY_S)

# Databricks notebook entrypoint
await produce()